# Figure 7.1: Wavelets over a finite field

*The Discrete Algebra of the Fourier Transform*, Gabriel Peyré.

Wavelets on $\FF_{53}$: the signal, the centered wavelet, and the magnitude of its affine-group transform. Translation $b$ is horizontal and nonzero dilation $a$ is vertical. Each row has its own indicated magnitude scale. The first signal has nonzero mean, which the mean-zero wavelet removes.

[Read this figure in the book](https://mathematical-tours.github.io/daft-sources/DAFT-EN.pdf#page=173).

## Prepare the example

Run the cells in order in Jupyter or Google Colab. The first cell installs missing Python packages and downloads the figure sources and input data once. Subsequent runs use the cached download. Python 3.10 or later is required.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import io
import os
import subprocess
import sys
import urllib.request
import zipfile

packages = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
            "PIL": "Pillow", "sympy": "sympy"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

archive_url = os.environ.get("DAFT_FIGURE_ARCHIVE", 'https://mathematical-tours.github.io/daft-sources/downloads/daft-figures-2e843bbe89f0.zip')
archive_sha = '2e843bbe89f0e274849b6896eb16bd9f7686521a8ca98f633240bfc06ea2d5b1'
workspace = Path(os.environ.get("DAFT_FIGURE_WORKDIR", "daft-figure-files")) / archive_sha[:12]
cache = workspace / "sources.zip"
workspace.mkdir(parents=True, exist_ok=True)
data = cache.read_bytes() if cache.exists() else urllib.request.urlopen(archive_url, timeout=60).read()
if hashlib.sha256(data).hexdigest() != archive_sha:
    raise ValueError("The figure-source download is incomplete or has changed. Download it again.")
cache.write_bytes(data)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    archive.extractall(workspace)
ROOT = (workspace / "daft-figures").resolve()
__file__ = str(ROOT / 'figures-code/chapter-linear-representations/finite-field-wavelets/generate.py')
print("Figure sources ready.")


## Generate the figure

This is the plotting code used by the book. Change its parameters and rerun this cell, then the display cell below, to explore the example.

In [ ]:
from pathlib import Path
import sys
ROOT=Path(__file__).resolve().parents[3]
sys.path.insert(0,str(ROOT/'tools'))
sys.path.insert(0,str(ROOT/'listings'))
from figure_style import *
from representations import finite_wavelet,inverse_finite_wavelet
p=53;x=np.arange(p);center=np.arange(-26,27)
f1=np.zeros(p);f1[24]=1
f2=np.r_[np.ones(8),np.zeros(p-16),-np.ones(8)]
f3=np.cos(6*np.pi*x/p)
p1=np.zeros(p);p1[[0,1,-1]]=[2,-1,-1]
p2=np.zeros(p);p2[[0,1,-1,-2]]=[1,1,-1,-1]
p3=np.zeros(p);p3[[0,1,-1,2,-2]]=[2,-1.2,-1.2,.2,.2]
fig,axes=plt.subplots(3,3,figsize=(8.5,7.1))
for row,(f,psi) in enumerate(zip([f1,f2,f3],[p1,p2,p3])):
    W=finite_wavelet(f,psi)
    np.testing.assert_allclose(inverse_finite_wavelet(W,psi),f-f.mean(),atol=1e-13)
    a,b,c=axes[row]
    a.plot(x,f,'o-',ms=2.4,lw=.7,color=TEAL)
    discrete(b,center,np.fft.fftshift(psi),color=RED)
    a.set(xlim=(-1,53),xticks=[0,13,26,39,52]);b.set(xlim=(-27,27),xticks=[-26,-13,0,13,26])
    for ax in [a,b]:
        ax.set_box_aspect(1);axes_box(ax,'residue $x$' if ax is a else 'centered residue $x$')
        ax.margins(y=.1)
    im=c.imshow(abs(W),cmap='gray_r',origin='upper',interpolation='nearest',vmin=0,
                extent=(-.5,p-.5,p-.5,.5),aspect='equal')
    c.set(xticks=[0,13,26,39,52],yticks=[1,13,26,39,52]);axes_box(c,'translation $b$','dilation $a$')
    colorbar=fig.colorbar(im,ax=c,shrink=.8,pad=.04,fraction=.045)
    colorbar.ax.tick_params(labelsize=8,length=2)
    if row==0:
        for ax,title in zip([a,b,c],['Signal $f$',r'Wavelet $\psi$',r'$|\mathcal{W}_\psi f(b,a)|$']):ax.set_title(title)
save(fig,'chapter-linear-representations','finite-field-wavelets')


## Display and save

In [ ]:
from IPython.display import Image, FileLink, display

display(Image(filename=str(ROOT / 'build/figure-previews/chapter-linear-representations/finite-field-wavelets.png')))
display(FileLink(os.path.relpath(ROOT / 'figures/chapter-linear-representations/finite-field-wavelets/figure.pdf', Path.cwd()),
                 result_html_prefix="Download the vector PDF: "))
